# AthleteAI — Notebook 1 : Extraction des fichiers .fit

## Contexte
Ce notebook constitue la première étape du projet AthleteAI. L'objectif est d'extraire les données brutes issues des fichiers `.fit` générés par ma montre GPS **Coros Pace 3** et de les transformer en un DataFrame exploitable pour les analyses suivantes.

Les fichiers `.fit` sont un format binaire propriétaire utilisé par la plupart des montres GPS sportives (Garmin, Coros, Polar). Ils contiennent des milliers de points de données enregistrés à chaque seconde d'entraînement : position GPS, fréquence cardiaque, vitesse, cadence, altitude, etc.

## Librairie utilisée
La librairie `fitparse` permet de lire et décoder ces fichiers binaires en Python.

In [1]:
import fitparse
import pandas as pd
import os
import glob

print("Librairies chargées ")

Librairies chargées 


## Étape 1 — Test sur un fichier unique

Avant de traiter l'ensemble des séances, on teste le parsing sur un seul fichier pour vérifier que la librairie fonctionne correctement et identifier les colonnes disponibles.

In [2]:
# Teste avec un seul fichier d'abord
sample_file = glob.glob("../data/raw/*.fit")[2]
print(f"Fichier test : {sample_file}")

fitfile = fitparse.FitFile(sample_file, check_crc=False)
records = []

for record in fitfile.get_messages("record"):
    data = {d.name: d.value for d in record}
    records.append(data)

df_sample = pd.DataFrame(records)
print(f"Colonnes disponibles : {list(df_sample.columns)}")
print(f"Nombre de points : {len(df_sample)}")
df_sample.head()

Fichier test : ../data/raw/470530566710657425.fit
Colonnes disponibles : ['activity_type', 'distance', 'timestamp', 'heart_rate', 'enhanced_speed', 'speed', 'cadence']
Nombre de points : 3220


,activity_type,distance,timestamp,heart_rate,enhanced_speed,speed,cadence
0,running,0.0,2025-07-18 17:28:43,NaN,NaN,NaN,NaN
1,running,0.0,2025-07-18 17:28:44,NaN,NaN,NaN,NaN
2,running,0.0,2025-07-18 17:28:45,NaN,NaN,NaN,NaN
3,running,0.0,2025-07-18 17:28:46,69.0,NaN,NaN,NaN
4,running,10.0,2025-07-18 17:28:47,69.0,4.167,4.167,NaN


Au début j'ai testé le premier fichier, mais à cause des problemes de formats de données affichées je suis partie sur le troisiéme juste pour avoir une idée claire des données pour le moment

## Étape 2 — Extraction complète

On applique la fonction de parsing à l'ensemble des fichiers `.fit` disponibles dans le dossier `data/raw/`. Chaque fichier correspond à une séance d'entraînement. Les fichiers corrompus ou vides sont ignorés automatiquement grâce au bloc `try/except`.

**Note technique** : Le paramètre `check_crc=False` est nécessaire pour les fichiers Coros dont certains champs ne respectent pas strictement le standard `.fit`.

In [6]:
def parse_fit_file(filepath):
    try:
        fitfile = fitparse.FitFile(filepath)
        records = []
        for record in fitfile.get_messages("record"):
            data = {d.name: d.value for d in record}
            records.append(data)
        df = pd.DataFrame(records)
        df["session_file"] = os.path.basename(filepath)
        return df
    except Exception as e:
        print(f"Erreur sur {filepath} : {e}")
        return None

# Charge tous les fichiers
all_files = glob.glob("../data/raw/*.fit")
print(f"{len(all_files)} fichiers .fit trouvés")

all_sessions = []
for f in all_files:
    df = parse_fit_file(f)
    if df is not None and len(df) > 0:
        all_sessions.append(df)

master_df = pd.concat(all_sessions, ignore_index=True)
print(f"Dataset total : {master_df.shape[0]} lignes, {master_df.shape[1]} colonnes")
master_df.head()

525 fichiers .fit trouvés
Erreur sur ../data/raw/467558234859798541.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/472083754924408935.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/470826534515540270.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/467282708947566607.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/472942938979074352.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/470297871015510022.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/473979041588740096.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/471204405467709443.fit : Invalid field size 1 for type 'uint32' (expected a multiple of 4)
Erreur sur ../data/raw/469068245771517960.fit : Invalid field 

,activity_type,distance,timestamp,Effort Pace,enhanced_speed,speed,heart_rate,altitude,enhanced_altitude,step_length,cadence,accumulated_power,power,position_lat,position_long,session_file
0,running,0.0,2024-11-17 22:47:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,464899027052298241.fit
1,running,0.0,2024-11-17 22:47:06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,464899027052298241.fit
2,running,0.0,2024-11-17 22:47:07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,464899027052298241.fit
3,running,0.0,2024-11-17 22:47:08,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,464899027052298241.fit
4,running,0.0,2024-11-17 22:47:09,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,464899027052298241.fit


## Étape 3 — Sauvegarde

Le DataFrame consolidé est sauvegardé en CSV dans `data/processed/master_sessions.csv` pour être utilisé dans les notebooks suivants. Ce fichier contient l'ensemble des points GPS/cardiaques de toutes les séances.

In [7]:
master_df.to_csv("../data/processed/master_sessions.csv", index=False)
print("Fichier sauvegardé  → data/processed/master_sessions.csv")

Fichier sauvegardé  → data/processed/master_sessions.csv


## Résultat

- **Fichiers parsés** : ensemble des séances disponibles
- **Colonnes extraites** : `timestamp`, `distance`, `heart_rate`, `enhanced_speed`, `speed`, `cadence`, `activity_type`
- **Fichier généré** : `data/processed/master_sessions.csv`

**Prochaine étape** : Notebook 2 — Analyse Exploratoire (EDA)

In [8]:
print(list(df_sample.columns))

['activity_type', 'distance', 'timestamp', 'heart_rate', 'enhanced_speed', 'speed', 'cadence']
